In [2]:
import os
import pandas as pd
import yaml
import random
from faker import Faker
fake = Faker('pt_BR')  # Brazilian locale for more realistic data

def load_config(file_path: str):
    """Carrega arquivo YAML"""
    with open(file_path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)

In [3]:
# Configura o diretório de trabalho para a raiz do projeto
import os
import sys

# Define o caminho da raiz do projeto diretamente
project_root = '/home/nicolas/Projects/Personal/corporate-benefits-transactions'
os.chdir(project_root)
print(f"Diretório de trabalho atual: {os.getcwd()}")

# Adiciona o diretório raiz ao path do Python para imports
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Carrega configurações usando caminhos relativos (agora que o diretório está correto)
business_rules = load_config('./config/business_rules.yaml')
volumes = load_config('./config/data_volumes.yaml')
print("Configurações carregadas com sucesso!")

Diretório de trabalho atual: /home/nicolas/Projects/Personal/corporate-benefits-transactions
Configurações carregadas com sucesso!


# DIMENSION TABLES

## 1. Municipalities

In [14]:
# Recarregando o módulo para garantir que temos a versão corrigida
import importlib
import src.extractors.municipality_data_importer
importlib.reload(src.extractors.municipality_data_importer)
from src.extractors.municipality_data_importer import generate_municipality_data

shapefile_path = './data/raw/municipios/municipios.shp'

municipalities = generate_municipality_data(shapefile_path)

## 8. Products

In [31]:
def generate_products(business_rules):
    """Generates products DataFrame based on business rules"""
    produtos = []
    for codigo, dados in business_rules['tipos_produto'].items():
      produto = {
          'product_id': str(f"PROD_{dados['code'].upper()}"),
          'code': dados['code'],
          'name': dados['name'],
          'category': dados['category'],
          'description': str(f"Produto de {dados['name']}"),
          'created_at': pd.Timestamp.now() 
      }
      produtos.append(produto)
    produtos = pd.DataFrame(produtos)
    produtos['code'] = produtos['code'].astype('category')
    produtos['name'] = produtos['name'].astype('category')
    produtos['category'] = produtos['category'].astype('category')

    return produtos

products = generate_products(business_rules)
products

,product_id,code,name,category,description,created_at
0,PROD_VA,VA,Vale Alimentação,Beneficio,Produto de Vale Alimentação,2025-08-20 11:35:00.071727
1,PROD_VR,VR,Vale Refeição,Beneficio,Produto de Vale Refeição,2025-08-20 11:35:00.071748
2,PROD_VC,VC,Vale Combustível,Beneficio,Produto de Vale Combustível,2025-08-20 11:35:00.071751
3,PROD_FR,FR,Gestão de Frota,Gestão,Produto de Gestão de Frota,2025-08-20 11:35:00.071754
4,PROD_GF,GF,Vale Presente,Beneficio,Produto de Vale Presente,2025-08-20 11:35:00.071756


## 2. CNAE

In [22]:
from src.extractors.kaggle_downloader import download_kaggle_dataset

def extract_cnae_data():
    download_kaggle_dataset("diovanimerlo/cnae-brazil-2024")

    cnaes = pd.read_csv(
        './data/raw/cnae_dataset_2024.csv',
        sep=',',
        encoding='utf-8',
        dtype={
            "secao_id": str,
            "secao": "category",
            "divisao_id": int,
            "divisao": "category",
            "grupo_id": int,
            "grupo": "category",
            "classe_id": int,
            "classe": "category",
            "subclasse_id": int,
            "subclasse": "category"
        }
    )
    return cnaes

cnaes = extract_cnae_data()

Path to dataset files: /home/nicolas/.cache/kagglehub/datasets/diovanimerlo/cnae-brazil-2024/versions/1
Arquivo já existe, pulando: /home/nicolas/Projects/Personal/corporate-benefits-transactions/src/extractors/../../data/raw/cnae_dataset_2024.csv


## 3. Segments

In [10]:
def generate_segments(business_rules):
    # Acessa a lista de segmentos dentro do dicionário
    segments_list = business_rules['segments']
    
    # Cria o DataFrame diretamente da lista carregada
    segments = pd.DataFrame(segments_list)
    
    # Adiciona a coluna 'created_at' após a criação do DataFrame para eficiência
    segments['created_at'] = pd.Timestamp.now()
    segments['name'] = segments['name'].astype('category')
    segments['category'] = segments['category'].astype('category')

    return segments
segments = generate_segments(business_rules)



## 4. Date (placeholder)

## 5. Acquirers

In [9]:
def generate_acquirers(business_rules):
    """Generates DataFrame for Acquirers based on business rules"""
    adquirentes = []
    for idx, name in enumerate(business_rules['adquirentes'], start=1):
        acquirer = {
            "acquirer_id": f"ACQ_{idx:03d}",
            "name": name,
            "active": True,
            "created_at": pd.Timestamp.now()
        }
        adquirentes.append(acquirer)

    return pd.DataFrame(adquirentes)

generate_acquirers(business_rules)

,acquirer_id,name,active,created_at
0,ACQ_001,Cielo,True,2025-08-20 11:19:37.479403
1,ACQ_002,Rede,True,2025-08-20 11:19:37.479422
2,ACQ_003,Stone,True,2025-08-20 11:19:37.479426
3,ACQ_004,GetNet,True,2025-08-20 11:19:37.479428
4,ACQ_005,PagSeguro,True,2025-08-20 11:19:37.479430
5,ACQ_006,SafraPay,True,2025-08-20 11:19:37.479432
6,ACQ_007,Adiq,True,2025-08-20 11:19:37.479433


## 6. Transaction Modalities

In [4]:
def generate_transaction_modalities(business_rules):
    """Generates DataFrame for Transaction Modalities based on business rules"""
    modalidades = []
    modalidades_yaml = business_rules['modalidades']
    # Se for dict, usa .items(); se for lista, itera direto
    if isinstance(modalidades_yaml, dict):
        iterator = modalidades_yaml.values()
    else:
        iterator = modalidades_yaml
    for idx, dados in enumerate(iterator, start=1):
        modality = {
            "modality_id": f"MOD_{idx:03d}",
            "name": dados["name"],
            "commission": dados.get("commission", 0.0),
            'created_at': pd.Timestamp.now()
        }
        modalidades.append(modality)
    return pd.DataFrame(modalidades)

transaction_modalities = generate_transaction_modalities(business_rules)
transaction_modalities

,modality_id,name,commission,created_at
0,MOD_001,Rede Credenciada,0.000,2025-08-20 11:19:03.930005
1,MOD_002,Visa,0.025,2025-08-20 11:19:03.930033
2,MOD_003,Mastercard,0.023,2025-08-20 11:19:03.930037
3,MOD_004,Elo,0.020,2025-08-20 11:19:03.930039


## 9. MCC Categories

In [8]:
import pandas as pd

def extract_mcc_data():
    mcc = pd.read_csv(
        './data/raw/mcc.csv',
        sep=',',
        encoding='utf-8',
        dtype={
            "mcc_id": str,
            "mcc_code": str,
            "mcc_description": str,
            "valid_payment_brands": str,
            "category": "category"
        }
    )
    return mcc

mcc_data = extract_mcc_data()
mcc_data

,mcc_id,mcc_code,mcc_description,valid_payment_brands,category
0,mcc_0742,0742,Veterinary Services,"V, M",veterinary
1,mcc_0763,0763,Agricultural Cooperatives,"V, M",agriculture
2,mcc_0780,0780,Horticultural and Landscaping Services,"V, M",landscaping
3,mcc_4111,4111,Transportation–Suburban and Local Commuter Pas...,"V, M",transportation
4,mcc_4112,4112,Passenger Railways,"V, M",transportation
...,...,...,...,...,...
261,mcc_9222,9222,Fines,"V, M",government
262,mcc_9223,9223,Bail and Bond Payments,"V, M",government
263,mcc_9311,9311,Tax Payments,"V, M",government
264,mcc_9399,9399,Government Services–Not Elsewhere Classified,"V, M",government


## 7. Capture Technologies

In [ ]:
def generate_capture_technologies():
    """Generate basic capture technologies"""
    technologies = ["POS", "Mobile", "Online", "QR Code", "NFC", "Manual"]
    return pd.DataFrame([
        {
            "capture_technology_id": f"CPT_{idx:03d}",
            "name": name,
            "active": True,
            "created_at": pd.Timestamp.now()
        }
        for idx, name in enumerate(technologies, start=1)
    ])

capture_technologies = generate_capture_technologies()
capture_technologies

# MAIN ENTITIES

## 1. Establishments 

In [ ]:
def generate_establishments(n_establishments, df_cnae, df_mcc, df_municipalities):
    """Generate establishments DataFrame based on existing CNAE, MCC and municipalities data"""
    establishments = []
    
    # Tipos de natureza jurídica mais comuns
    legal_natures = ["LTDA", "EIRELI", "SA", "MEI", "EPP", "COOPERATIVA"]
    company_sizes = ["MEI", "ME", "EPP", "PEQUENO PORTE", "MEDIO PORTE", "GRANDE PORTE"]
    statuses = ["ATIVA", "INATIVA", "SUSPENSA", "BAIXADA"]
    status_reasons = ["NORMAL", "IRREGULARIDADE", "FALTA DE MOVIMENTO", "MUDANÇA DE ENDEREÇO"]
    
    for i in range(n_establishments):
        establishment = {
            'establishment_id': f"EST_{i:06d}",
            'cnpj': fake.cnpj(),
            'legal_name': fake.company(),
            'trade_name': fake.company_suffix() + " " + fake.last_name(),
            'main_cnae_id': random.choice(df_cnae['subclasse_id'].tolist()),
            'mcc_code': random.choice(df_mcc['mcc_code'].tolist()),
            'legal_nature': random.choice(legal_natures),
            'company_size': random.choice(company_sizes),
            'municipality_id': random.choice(df_municipalities['municipio_id'].tolist()),
            'postal_code': fake.postcode(),
            'street': fake.street_name(),
            'number': str(fake.building_number()),
            'complement': random.choice([None, "Sala " + str(random.randint(1, 100)), "Loja " + str(random.randint(1, 50))]),
            'neighborhood': fake.neighborhood(),
            'email': fake.company_email(),
            'phone': fake.phone_number(),
            'registration_status': random.choice(statuses),
            'status_date': fake.date_between(start_date='-2y', end_date='today'),
            'status_reason': random.choice(status_reasons),
            'opening_date': fake.date_between(start_date='-10y', end_date='-1y'),
            'created_at': pd.Timestamp.now()
        }
        establishments.append(establishment)
    
    return pd.DataFrame(establishments)

# Gerar dados de establishments usando o volume configurado
establishments_df = generate_establishments(volumes['volumes']['empresas'], cnaes, mcc_data, municipalities)
establishments_df

## 2. Companies

In [16]:
def generate_companies(n_companies, segments):
    companies = []
    for i in range(n_companies):
        company = {
            'company_id': f"EMP_{i}",
            'parent_company_id': random.choice([None] + [c["company_id"] for c in companies]) if companies else None,
            'name': fake.company(),
            'cnpj': fake.cnpj(),
            'company_type': random.choice(["Privada", "Publica"]),
            'segment_id': random.choice(segments['segment_id'].tolist()),
            'municipality_id': random.choice(municipalities['municipality_id'].tolist()),
            'contract_registration_date': fake.date_between(start_date='-5y', end_date='today'),
            'created_at': pd.Timestamp.now()
        }
        companies.append(company)
    return pd.DataFrame(companies)  # Return as DataFrame for consistency

# Fixed the volumes access - need to access volumes['volumes']['empresas']
companies_df = generate_companies(volumes['volumes']['empresas'], segments)
companies_df

,company_id,parent_company_id,name,cnpj,company_type,segment_id,municipality_id,contract_registration_date,created_at
0,EMP_0,None,Borges,16.548.237/0001-41,Publica,SEG_05,4125001,2024-12-18,2025-08-20 11:20:36.275030
1,EMP_1,None,Ferreira,54.689.027/0001-62,Privada,SEG_03,3113107,2020-11-15,2025-08-20 11:20:36.275442
2,EMP_2,EMP_0,Pastor,60.839.217/0001-11,Privada,SEG_04,2313955,2022-12-15,2025-08-20 11:20:36.275835
3,EMP_3,EMP_2,Cardoso da Cunha - EI,17.402.598/0001-48,Privada,SEG_07,2900900,2022-02-22,2025-08-20 11:20:36.276203
4,EMP_4,EMP_3,Sá Câmara - EI,37.062.195/0001-04,Publica,SEG_05,4124400,2023-06-22,2025-08-20 11:20:36.276577
5,EMP_5,None,Abreu - ME,18.295.036/0001-05,Publica,SEG_10,1703883,2021-01-14,2025-08-20 11:20:36.276947
6,EMP_6,None,Vasconcelos,54.076.893/0001-88,Privada,SEG_04,3109451,2020-08-24,2025-08-20 11:20:36.277195
7,EMP_7,EMP_6,Macedo S.A.,91.425.706/0001-50,Publica,SEG_06,1700251,2022-05-23,2025-08-20 11:20:36.277419
8,EMP_8,EMP_2,Ferreira,32.647.108/0001-67,Privada,SEG_10,2802502,2022-02-10,2025-08-20 11:20:36.277672
9,EMP_9,EMP_5,Fonseca Pastor - ME,36.824.510/0001-20,Publica,SEG_14,3530508,2023-01-22,2025-08-20 11:20:36.278134


## 3. Cardholders (TODO)  

In [17]:
def generate_cardholders(n_funcionarios, companies_df, municipalities_df):
    """Generate cardholders DataFrame based on existing companies data"""
    cardholders = []
    
    for i in range(n_funcionarios):
        cardholder = {
            'cardholder_id': f"FUNC_{i}",
            'company_id': random.choice(companies_df['company_id'].tolist()),
            'name': fake.name(),
            'cpf': fake.cpf(),
            'municipality_id': random.choice(municipalities_df['municipality_id'].tolist()),
            'admission_date': fake.date_this_decade(),
            'position': fake.job(),
            'department': fake.word(),
            'status': random.choice(['active', 'inactive', 'terminated']),
            'created_at': pd.Timestamp.now()
        }
        cardholders.append(cardholder)
    
    return pd.DataFrame(cardholders)

cardholders = generate_cardholders(volumes['volumes']['funcionarios'], companies_df, municipalities)
cardholders

,cardholder_id,company_id,name,cpf,municipality_id,admission_date,position,department,status,created_at
0,FUNC_0,EMP_3,Helena Borges,374.196.580-48,4113429,2022-09-13,Tecnólogo em papel e celulose,autem,active,2025-08-20 11:20:38.630821
1,FUNC_1,EMP_3,Cauã Araújo,035.821.794-60,3203163,2025-06-09,Acompanhante,quo,terminated,2025-08-20 11:20:38.631105
2,FUNC_2,EMP_6,Bryan Nogueira,453.602.891-98,2904506,2021-04-11,Artesão,consectetur,inactive,2025-08-20 11:20:38.631377
3,FUNC_3,EMP_4,Srta. Milena Azevedo,459.317.820-79,2111805,2022-01-17,Almoxarife,nulla,active,2025-08-20 11:20:38.631648
4,FUNC_4,EMP_5,Carolina da Paz,926.873.105-30,3300159,2022-01-30,Despachante,consequatur,inactive,2025-08-20 11:20:38.631908
...,...,...,...,...,...,...,...,...,...,...
795,FUNC_795,EMP_0,Helena Correia,058.132.694-60,3144003,2020-08-11,Ambulante,omnis,terminated,2025-08-20 11:20:38.806622
796,FUNC_796,EMP_5,Calebe Pereira,547.820.631-80,2112209,2023-06-06,Empacotador,reprehenderit,terminated,2025-08-20 11:20:38.806834
797,FUNC_797,EMP_4,Daniela Freitas,928.147.306-22,3300159,2022-04-06,Motorista,consequuntur,inactive,2025-08-20 11:20:38.807042
798,FUNC_798,EMP_2,Lucas Gabriel Santos,674.982.103-04,4123501,2021-11-11,Defensor Público,repellat,inactive,2025-08-20 11:20:38.807249


## 4. Cards (TODO)

In [ ]:
def generate_cards(n_cards, cardholders_df, modality_df):
    cards = []

    for idx in range(n_cards):
        card_number = "".join([str(random.randint(0, 9)) for _ in range(16)])
        formatted_card_number = " ".join([card_number[i:i+4] for i in range(0, 16, 4)])
        
        cvv = "".join([str(random.randint(0, 9)) for _ in range(3)])

        card = {
            "card_id": idx,
            "cardholder_id": random.choice(cardholders_df['cardholder_id'].tolist()),
            "card_number": formatted_card_number,
            "modality_id": random.choice(modality_df['modality_id'].tolist()),
            "expiration_date": fake.date_between(start_date='today', end_date='+5y'),
            "cvv": cvv,
            "card_type": random.choice(['credit', 'debit']),
            "issuance_date": pd.Timestamp.now(),
            "status": random.choice(['active', 'inactive']),
            "created_at": pd.Timestamp.now()
        }
        cards.append(card)

    return pd.DataFrame(cards)

cards = generate_cards(volumes['volumes']['cartoes'], cardholders, transaction_modalities)
cards


## 5. Card Products (TODO)

In [ ]:
def generate_card_products(cards_df, products_df, min_products=1, max_products=3):
    """
    Gera produtos associados aos cartões.
    Cada cartão pode ter entre min_products e max_products produtos.
    """
    card_products = []

    for card_id in cards_df['card_id'].tolist():
        n_products = random.randint(min_products, max_products)

        for _ in range(n_products):
            activation_date = fake.date_between(start_date='-2y', end_date='today')

            # 30% de chance de já ter sido desativado
            if random.random() < 0.3:
                deactivation_date = fake.date_between(start_date=activation_date, end_date='today')
                status = random.choice(['suspended', 'cancelled'])
            else:
                deactivation_date = None
                status = 'active'

            card_product = {
                "card_product_id": fake.uuid4(),
                "card_id": card_id,
                "product_id": random.choice(products_df['product_id'].tolist()),
                "current_balance": round(random.uniform(0, 5000), 2),
                "daily_limit": round(random.uniform(100, 2000), 2),
                "monthly_limit": round(random.uniform(1000, 10000), 2),
                "activation_date": activation_date,
                "deactivation_date": deactivation_date,
                "status": status,
                "created_at": pd.Timestamp.now()
            }
            card_products.append(card_product)

    return pd.DataFrame(card_products)

# uso
card_products = generate_card_products(cards, products, min_products=1, max_products=3)
card_products
